In [ ]:
# Quantum ESPRESSO
# https://ase-lib.org/ase/calculators/espresso.html#
from ase.calculators.espresso import Espresso, EspressoProfile
from ase.visualize import view
    
from ase.build import bulk

# -----------------------------------------------------------------------------------------

# https://docs.matlantis.com/atomistic-simulation-tutorial/en/Appendix_1_visualization.html
# conda install conda-forge::nglview
import nglview as nv 

## SCF INPUT

In [6]:
from ase.build import bulk
from ase.calculators.espresso import Espresso, EspressoProfile

# 1. Definição da Estrutura
Zinc_Oxide = bulk("ZnO", crystalstructure="wurtzite", a=3.25, c=5.21)

# 2. Caminhos essenciais (agora separados do input_data)
pseudo_dir = "/home/jvc/ZnO_database/pseudos/ppdojo_LDA/"
pw_exec = "/home/jvc/anaconda3/envs/dft_ase/bin/pw.x"

# O ASE antigo prefere o comando inteiro como uma string
comando_mpi = f"mpirun -np 16 {pw_exec}"

# 3. Criando o perfil com os argumentos obrigatórios solicitados pelo erro
profile = EspressoProfile(command=comando_mpi, pseudo_dir=pseudo_dir)

# 4. Configuração dos parâmetros do Quantum ESPRESSO
input_data = {
    "control": {
        "calculation": "scf",
        "prefix": "ZnO_LDA",
        "outdir": "./",
        "disk_io": "none",
        "verbosity": "high",
        # pseudo_dir foi removido daqui pois o profile já vai cuidar dele
    },
    "system": {
        "ibrav": 0, 
        "ecutwfc": 80, 
        "ecutrho": 320, 
        "occupations": "fixed"
    },
    "electrons": {
        "conv_thr": 1.0e-8, 
        "mixing_beta": 0.3
    },
}

pseudopotentials = {
    "Zn": "Zn_pseudo-dojo_NC_SR_LDA.upf",
    "O": "O_pseudo-dojo_NC_SR_LDA.upf",
}

# 5. Instanciação do Calculador
calc = Espresso( 
    profile=profile,
    pseudopotentials=pseudopotentials,
    input_data=input_data,
    kpts=(6, 6, 4),
    tstress=True,
    tprnfor=True
)

# 6. Execução
Zinc_Oxide.calc = calc
print("Iniciando o cálculo SCF...")
energia_total = Zinc_Oxide.get_potential_energy()

print(f"Cálculo finalizado! Energia total: {energia_total:.4f} eV")

Iniciando o cálculo SCF...
Cálculo finalizado! Energia total: -12270.0415 eV


In [8]:
Zinc_Oxide.get_forces()

array([[ 0.        ,  0.        ,  0.02704158],
       [-0.        ,  0.        , -0.02704158],
       [ 0.        ,  0.        ,  0.02704158],
       [ 0.        , -0.        , -0.02704158]])